# Tome 13, Chapter 1: Zeros and primes in short intervals (Python companion)

Exercises first, solutions at the end. We look at the gaps between primes up to $10^8$, test the mean value theorem for Dirichlet polynomials, watch a mollifier make $\zeta$ close to $1$, and turn to a computer science question the gaps answer: **how much work does it take to generate a random prime for RSA**, and how far from uniform is the prime we get? Requires `numpy`, `mpmath` and `sympy`.

In [ ]:
import math, random
import numpy as np
import mpmath as mp
from sympy import isprime, mobius
rng = np.random.default_rng(13)
random.seed(13)

## Exercises

**P1.1 (★) Record gaps.** Sieve the primes up to $10^8$ and list the record gaps $g=p_{n+1}-p_n$ (gaps longer than all earlier ones). For each record, print $g$, $\frac g{(\log p)^2}$ and $\frac g{\sqrt p\log p}$. Which bound is closer to the truth: Cramér's conjecture $O((\log p)^2)$ or the Riemann hypothesis bound $O(\sqrt p\log p)$?

**P1.2 (★) The mean value theorem.** For $M=500$ and random complex coefficients $a_n$ on $]M,2M]$, compute $\frac1T\int_0^T|\sum a_nn^{-it}|^2\,dt$ divided by $\sum|a_n|^2$, for $T=10,100,1000,10^4$. Compare with $1+\frac{8M(1+\log M)}T$ (Lemma 1.3). Then take all $a_n=1$: what changes for small $T$, and why (Exercise 1.2)?

**P1.3 (★★) A mollifier at work.** For $T=500$ and $\sigma\in\{0.6,0.75,1\}$, estimate $\frac1T\int_T^{2T}|\zeta(\sigma+it)M_X(\sigma+it)-1|^2\,dt$ on a grid of $400$ points, for $X=1,5,25,125$. (For $X=1$ the integrand is $|\zeta-1|^2$.) How does the mean square depend on $X$ and on $\sigma$? Compare with the prediction $\sum_{n>X}d(n)^2n^{-2\sigma}$ of the diagonal terms, roughly $X^{1-2\sigma}$.

**P1.4 (★★) CS application: generating RSA primes.** A common way to generate a random prime of a given size is: choose a random odd $x$, and test $x,x+2,x+4,\dots$ until a prime is found.

(a) With the primes below $10^8$, compute the expected number of candidates tested when $x$ is a uniform odd number in $[5\cdot10^7,10^8]$, exactly from the gaps: a gap $g$ is hit by $g/2$ odd starting points, which then need on average $\frac{g/2+1}2$ tests. Compare with $\frac{\log x}2$, the answer if gaps were exponential with mean $\log x$ (Cramér's model), and with the naive $\frac{\log x}4$ that half the mean gap would suggest.

(b) The prime returned is $p$ with probability proportional to the gap $g$ *before* $p$. How unequal are these probabilities: what are the largest and smallest gaps in the range, and how much entropy (in bits) is lost compared with a uniform choice among the primes in the range?

(c) Generate $300$ random $256$-bit primes this way (with `sympy.isprime`) and compare the average number of candidates with (a)'s formula at $x=2^{256}$.

## Solutions

### P1.1
**How to find it.** A sieve of the odd numbers in a byte array fits in memory for $10^8$; the gaps are the differences of the list of primes.

In [ ]:
L = 10**8
s = np.ones(L//2, dtype=bool); s[0] = False            # s[i] <-> 2i+1
for i in range(1, int(math.isqrt(L))//2 + 1):
    if s[i]:
        p = 2*i + 1
        s[p*p//2::p] = False
primes = np.concatenate(([2], 2*np.nonzero(s)[0] + 1))
gaps = np.diff(primes)
print(len(primes), "primes below 10^8")
best = 0
print(f"{'p':>11} {'gap':>5} {'g/log^2 p':>10} {'g/(sqrt p log p)':>17}")
for i in range(len(gaps)):
    if gaps[i] > best:
        best = gaps[i]; p = int(primes[i]); lp = math.log(p)
        print(f"{p:>11} {best:>5} {best/lp**2:>10.3f} {best/(math.sqrt(p)*lp):>17.5f}")

**Write-up.** From $p=23$ on, every record gap is below $(\log p)^2$: the ratio $g/(\log p)^2$ stays between $0.39$ and $0.74$ (the largest is $0.739$, for the gap of $210$ after $20\,831\,323$). The ratio to $\sqrt p\log p$ collapses, below $0.002$ near $5\cdot10^7$: even the Riemann hypothesis bound is very far from the truth, and the unconditional $p^{0.525}$ further still. Cramér's model is much closer; Granville argued in 1995 ("Harald Cramér and the distribution of prime numbers", *Scand. Actuarial J.*) that the true constant in Cramér's conjecture should be at least $2e^{-\gamma}\approx1.12$, larger than Cramér's $1$, because the small primes make some intervals much poorer than the model says.

### P1.2
**How to find it.** $\int_0^T|F|^2$ for a polynomial of length $500$ can be computed exactly from $\sum_{m,n}a_m\overline{a_n}\frac{e^{iT\log(n/m)}-1}{i\log(n/m)}$, the expansion used in the proof of Lemma 1.3.

In [ ]:
M = 500
n = np.arange(M+1, 2*M+1)
lg = np.log(n)
D = lg[:, None] - lg[None, :]                                    # log(m/n)
def mean_square(a, T):
    with np.errstate(divide='ignore', invalid='ignore'):
        K = np.where(D == 0, T, (np.exp(1j*T*D) - 1)/(1j*D))     # int_0^T (m/n)^{it} dt
    return (np.conj(a) @ K @ a).real / T / np.sum(abs(a)**2)     # int |sum a_n n^{-it}|^2 / (T sum|a|^2)
a = rng.standard_normal(M) + 1j*rng.standard_normal(M)
one = np.ones(M)
for T in [10, 100, 1000, 10**4]:
    print(f"T={T:>6}: random a {mean_square(a, T):.3f}   all ones {mean_square(one, T):8.2f}   "
          f"bound 1+8M(1+log M)/T = {1 + 8*M*(1 + math.log(M))/T:9.1f}")

**Write-up.** With random coefficients the ratio is between $0.7$ and $1$ already for small $T$: the off-diagonal terms cancel among themselves, not because $T$ is large. With $a_n=1$ the terms add coherently near $t=0$: for $T=10$ the ratio is about $210$, and it decreases like $\frac{cM}T$ (the sum $\sum n^{-it}$ is close to $\int_M^{2M}u^{-it}du$, of size $\frac M{|t|}$, so its mean square over $[0,T]$ is of order $\frac{M^2}{T}$). The lemma's bound, with its $8M(1+\log M)$, is pessimistic by a factor of more than ten here, but its shape $T+O(M)$ is right, as Exercise 1.2 shows.

### P1.3
**How to find it.** `mpmath.zeta` evaluates $\zeta$ anywhere; $M_X$ is a short sum. The diagonal prediction comes from the formal expansion $\zeta M_X-1=\sum_{n>X}b_nn^{-s}$.

In [ ]:
mp.mp.dps = 15
T = 500
ts = np.linspace(T, 2*T, 400)
mu = {d: int(mobius(d)) for d in range(1, 126)}
def MX(s, X):
    return sum(mu[d]*mp.power(d, -s) for d in range(1, X+1) if mu[d])
def b(nmax, X):
    bb = np.zeros(nmax+1)
    for d in range(1, X+1):
        if mu[d]: bb[d::d] += mu[d]
    return bb
bb = {X: b(200000, X) for X in [1, 5, 25, 125]}
idx = np.arange(200001)
for sig in [0.6, 0.75, 1.0]:
    zs = [mp.zeta(mp.mpc(sig, t)) for t in ts]
    row = []
    for X in [1, 5, 25, 125]:
        vals = [abs(z*MX(mp.mpc(sig, t), X) - 1)**2 for z, t in zip(zs, ts)]
        diag = np.sum(bb[X][2:]**2 * idx[2:]**(-2*sig))
        row.append(f"X={X:>3}: {float(np.mean(vals)):7.4f} (diag {diag:.4f})")
    print(f"sigma={sig}:  " + "   ".join(row))

**Write-up.** At $\sigma=1$ the mean square drops sharply with $X$, and closely follows the diagonal prediction: the mollifier does what it was built for. At $\sigma=0.75$ it still decreases, but more slowly, by a factor of about $2.7$ each time $X$ is multiplied by $5$ (the rough prediction $X^{1-2\sigma}=X^{-1/2}$ gives $2.2$). At $\sigma=0.6$ it decreases more slowly still, and at $\sigma=\frac12$ the gain would vanish, since $\sum_nd(n)^2n^{-1}$ diverges: the diagonal sum is truncated at $2\cdot10^5$ in the code, which is why it stays finite. This is the trade-off of Proposition 1.6: a longer mollifier helps on the right and costs on the left, and Carlson's exponent is the best balance. (At $\sigma=0.6$ the diagonal prediction is only roughly met, because the off-diagonal terms over a range of $500$ heights are not yet negligible.)

### P1.4
**How to find it.** A uniform odd starting point falls in the gap $]p,p+g]$ with probability proportional to $g/2$; the number of odd candidates it then tests is uniform on $1,\dots,g/2$, of mean $\frac{g/2+1}2$. Summing over gaps gives the expectation exactly.

In [ ]:
lo, hi = 5*10**7, 10**8
sel = (primes[:-1] >= lo) & (primes[1:] <= hi)
g = gaps[sel].astype(float)
w = g/2                                         # odd starting points per gap
expected = np.sum(w*(w + 1)/2) / np.sum(w)
x = 7.5e7
print(f"(a) expected candidates {expected:.2f};  Cramer model log(x)/2 = {math.log(x)/2:.2f};  naive log(x)/4 = {math.log(x)/4:.2f}")
P = g/np.sum(g)                                  # probability that the prime after each gap is returned
H = -np.sum(P*np.log2(P)); H0 = math.log2(len(g))
print(f"(b) gaps from {int(g.min())} to {int(g.max())}; entropy {H:.4f} bits vs uniform {H0:.4f}: loss {H0 - H:.3f} bits")
def random_prime(bits):
    x = random.getrandbits(bits) | (1 << (bits - 1)) | 1
    k = 1
    while not isprime(x):
        x += 2; k += 1
    return k
ks = [random_prime(256) for _ in range(300)]
print(f"(c) 256 bits: mean candidates {np.mean(ks):.1f} +- {np.std(ks)/math.sqrt(len(ks)):.1f};  log(2^256)/2 = {256*math.log(2)/2:.1f}")

**Write-up.** (a) The expected number of candidates is close to $\frac{\log x}2\approx9.1$, not $\frac{\log x}4$: the starting point falls more often in long gaps (the *inspection paradox*), and for exponential gaps the size-biased mean is exactly twice the mean. The data are slightly below the Cramér value, because small gaps are a little more frequent than the model predicts (the primes avoid multiples of $3$, $5$, $7$, which spreads them out more evenly).

(b) Gaps in the range vary from $2$ to over $200$, so some primes are a hundred times more likely to be returned than others. Yet the entropy loss is well under one bit: a few primes are favoured, but there are millions of them, and the distribution is still close to uniform in the sense that matters for key generation. (The loss of a fraction of a bit is the price of the *incremental search*, which is cheap and widely used; a uniform prime is obtained by rejecting and redrawing $x$ instead, at the cost of more random bits.)

(c) For $256$-bit primes the average is about $81$ candidates, about $1.7$ standard errors below $\frac{\log x}2$, slightly below as in (a). The theorems of this chapter guarantee much less: the search could in principle need $x^{0.525}$ steps. That it never does in practice is exactly Cramér's conjecture at work.